# 클래스 불균형 방식에 따라 달라지는 성능평가 결과 비교

## XGB, LightGBM

## Class Weight

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import copy

from imblearn.over_sampling import BorderlineSMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5

YEAR_COL = "회계년도"
ID_COLS  = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012

PRIMARY_METRICS = ["F1", "Recall", "ROC_AUC", "Precision", "PR_AUC", "Accuracy"]



# ============================================================
# 2. 피처 파일 (★ 3개 파일로 확장)
# ============================================================

feature_files = {
    "top50_dedup43": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top50--43.csv",
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
}


# ============================================================
# 3. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

train_ids = train_full[[c for c in ID_COLS if c in train_full.columns]].reset_index(drop=True)
test_ids  = test[[c for c in ID_COLS if c in test.columns]].reset_index(drop=True)

print("=" * 70)
print(f"Train shape: {train_full.shape} | Test shape: {test.shape}")
print("=" * 70)


# ============================================================
# 4. 피처 목록 로드 & 유효성 확인
# ============================================================

feature_map = {}   # feature_name → 실제 사용 피처 리스트

for feature_name, feature_path in feature_files.items():
    df_feat = pd.read_csv(feature_path)
    # 헤더명이 'feature' 또는 'column' 둘 다 대응
    col_key = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features  = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  파일 내 피처: {len(raw_features)}개  "
          f"→ 실제 사용(train 컬럼 교집합): {len(valid_features)}개")

print("=" * 70)


# ============================================================
# 5. pos_weight 계산
# ============================================================

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()
print(f"XGBoost scale_pos_weight : {pos_weight:.4f}")
print(f"클래스 분포 → 정상(0): {(y_train_full==0).sum()}, 부실(1): {(y_train_full==1).sum()}")
print("=" * 70)


# ============================================================
# 6. 모델 군 정의
# ============================================================


models_cw = {

    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )
}


# ============================================================
# 7. 오버샘플링 함수
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    return X_train.copy(), y_train.copy()


def apply_borderline_smote(X_train, y_train, ratio):
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    return (
        pd.DataFrame(X_res, columns=X_train.columns),
        pd.Series(y_res, name=TARGET_COL)
    )


# ============================================================
# 8. method_configs
# ============================================================

method_configs = {
    # "None": (
    #     apply_none,
    #     [None],
    #     models_base
    # ),
    "ClassWeight": (
        apply_none,
        [None],
        models_cw
    ),
    # "BorderlineSMOTE": (
    #     apply_borderline_smote,
    #     [0.2, 0.3],
    #     models_base
    # ),
    # "ClassWeight+BorderlineSMOTE": (
    #     apply_borderline_smote,
    #     [0.2, 0.3],
    #     models_cw
    # ),
}


# ============================================================
# 9. 평가 지표 계산 헬퍼
# ============================================================

def calc_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 10. Expanding Window CV 인덱스 생성
# ============================================================

def make_expanding_folds(df, year_col, fold_val_years, train_start):
    folds = []
    for val_year in fold_val_years:
        train_idx = df.index[
            (df[year_col] >= train_start) & (df[year_col] < val_year)
        ]
        val_idx = df.index[df[year_col] == val_year]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx))
        else:
            print(f"  [경고] val_year={val_year} fold 생성 불가 (데이터 없음)")
    return folds


# ============================================================
# 11. 메인 실험 루프
# ============================================================

cv_results   = []
test_results = []

for feature_name, use_features in feature_map.items():

    # ★ 피처 파일명 & 피처 수 메타 정보
    feature_file_name = feature_files[feature_name].split("\\")[-1]  # 파일명만
    n_features        = len(use_features)

    print(f"\n{'='*70}")
    print(f"[Feature Set: {feature_name}]  파일: {feature_file_name}  |  피처 수: {n_features}개")
    print(f"{'='*70}")

    folds = make_expanding_folds(train_full, YEAR_COL, FOLD_VAL_YEARS, TRAIN_START)
    print(f"  생성된 fold 수: {len(folds)}")

    for method_name, (oversample_fn, ratios, models_dict) in method_configs.items():

        print(f"\n  {'='*60}")
        print(f"  방식: {method_name}")

        for ratio in ratios:
            ratio_label = ratio if ratio is not None else "-"
            print(f"\n    ratio = {ratio_label}")

            # ── (A) Expanding Window CV ───────────────────────────
            for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):

                X_fold_train = train_full.loc[train_idx, use_features]
                y_fold_train = train_full.loc[train_idx, TARGET_COL]
                X_fold_val   = train_full.loc[val_idx,   use_features]
                y_fold_val   = train_full.loc[val_idx,   TARGET_COL]
                val_year     = FOLD_VAL_YEARS[fold_idx - 1]

                # ── NaN imputation (SMOTE 이전, train fold에만 fit) ──
                imputer        = SimpleImputer(strategy="median")
                X_fold_train   = pd.DataFrame(
                    imputer.fit_transform(X_fold_train), columns=use_features
                )
                X_fold_val_imp = pd.DataFrame(
                    imputer.transform(X_fold_val), columns=use_features
                )

                X_res, y_res = oversample_fn(X_fold_train, y_fold_train, ratio)

                n0 = (y_res == 0).sum()
                n1 = (y_res == 1).sum()

                # ── LR용 스케일링 (imputed 데이터 기준) ──────────
                scaler       = StandardScaler()
                X_res_scaled = pd.DataFrame(
                    scaler.fit_transform(X_res), columns=use_features
                )
                X_val_scaled = pd.DataFrame(
                    scaler.transform(X_fold_val_imp), columns=use_features
                )

                for model_name, model in models_dict.items():
                    m = copy.deepcopy(model)

                    if model_name == "LogisticRegression":
                        m.fit(X_res_scaled, y_res)
                        y_prob_val = m.predict_proba(X_val_scaled)[:, 1]
                    else:
                        # 트리 모델도 imputed 데이터 사용
                        m.fit(X_res, y_res)
                        y_prob_val = m.predict_proba(X_fold_val_imp)[:, 1]

                    val_metrics = calc_metrics(y_fold_val, y_prob_val)

                    cv_results.append({
                        "FeatureSet"      : feature_name,        # ★ 추가
                        "FeatureFile"     : feature_file_name,   # ★ 추가
                        "N_Features"      : n_features,          # ★ 추가
                        "Method"          : method_name,
                        "SMOTE_Ratio"     : ratio_label,
                        "Model"           : model_name,
                        "Fold"            : fold_idx,
                        "Val_Year"        : val_year,
                        "Train_N0"        : n0,
                        "Train_N1"        : n1,
                        **{f"Val_{k}": v for k, v in val_metrics.items()},
                    })

            # ── (B) Test 평가: 전체 train으로 재학습 ─────────────
            X_train_all = train_full[use_features]
            y_train_all = train_full[TARGET_COL]

            # ── NaN imputation (SMOTE 이전, 전체 train에만 fit) ──
            imputer_final = SimpleImputer(strategy="median")
            X_train_all   = pd.DataFrame(
                imputer_final.fit_transform(X_train_all), columns=use_features
            )
            X_test_imp    = pd.DataFrame(
                imputer_final.transform(test[use_features]), columns=use_features
            )

            X_tr_res, y_tr_res = oversample_fn(X_train_all, y_train_all, ratio)

            # ── LR용 스케일링 ─────────────────────────────────────
            scaler_final    = StandardScaler()
            X_tr_res_scaled = pd.DataFrame(
                scaler_final.fit_transform(X_tr_res), columns=use_features
            )
            X_test_scaled   = pd.DataFrame(
                scaler_final.transform(X_test_imp), columns=use_features
            )

            for model_name, model in models_dict.items():
                m = copy.deepcopy(model)

                if model_name == "LogisticRegression":
                    m.fit(X_tr_res_scaled, y_tr_res)
                    y_prob_test = m.predict_proba(X_test_scaled)[:, 1]
                else:
                    # 트리 모델도 imputed 데이터 사용
                    m.fit(X_tr_res, y_tr_res)
                    y_prob_test = m.predict_proba(X_test_imp)[:, 1]

                test_metrics = calc_metrics(y_test, y_prob_test)

                test_results.append({
                    "FeatureSet"      : feature_name,        # ★ 추가
                    "FeatureFile"     : feature_file_name,   # ★ 추가
                    "N_Features"      : n_features,          # ★ 추가
                    "Method"          : method_name,
                    "SMOTE_Ratio"     : ratio_label,
                    "Model"           : model_name,
                    **{f"Test_{k}": v for k, v in test_metrics.items()},
                })

                print(
                    f"      [{model_name}]  "
                    f"Test F1={test_metrics['F1']:.4f} | "
                    f"Recall={test_metrics['Recall']:.4f} | "
                    f"PR_AUC={test_metrics['PR_AUC']:.4f}"
                )


# ============================================================
# 12. 결과 DataFrame 변환
# ============================================================

cv_df   = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)

# ============================================================
# 13. CV 평균 집계 + Test 결과 병합 → summary_df
# ============================================================

cv_agg = (
    cv_df
    .groupby(["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"])
    [[ f"Val_{m}" for m in PRIMARY_METRICS]]
    .mean()
    .reset_index()
)

summary_df = cv_agg.merge(
    test_df,
    on=["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"],
    how="left"
)

# CV-Test 갭 계산 (과적합/과소적합 진단용)
for m in PRIMARY_METRICS:
    summary_df[f"Gap_{m}"] = summary_df[f"Val_{m}"] - summary_df[f"Test_{m}"]

print(f"\nsummary_df shape: {summary_df.shape}")
print(summary_df[["Method", "Model", "Val_F1", "Test_F1", "Gap_F1"]].to_string(index=False))


# ============================================================
# 14. 저장
# ============================================================

summary_df.to_csv(
    "Summary_CV_Test_MultiFeature_try2.csv",
    index=False, encoding="utf-8-sig"
)


Train shape: (28111, 256) | Test shape: (11797, 256)
[top50_dedup43]  파일 내 피처: 43개  → 실제 사용(train 컬럼 교집합): 43개
[top55_dedup45]  파일 내 피처: 45개  → 실제 사용(train 컬럼 교집합): 45개
XGBoost scale_pos_weight : 25.6455
클래스 분포 → 정상(0): 27056, 부실(1): 1055

[Feature Set: top50_dedup43]  파일: lasso_features_top50--43.csv  |  피처 수: 43개
  생성된 fold 수: 6

  방식: ClassWeight

    ratio = -
      [XGBoost]  Test F1=0.4340 | Recall=0.8722 | PR_AUC=0.4525

[Feature Set: top55_dedup45]  파일: lasso_features_top55--45.csv  |  피처 수: 45개
  생성된 fold 수: 6

  방식: ClassWeight

    ratio = -
      [XGBoost]  Test F1=0.4345 | Recall=0.8656 | PR_AUC=0.4567

summary_df shape: (2, 24)
     Method   Model   Val_F1  Test_F1    Gap_F1
ClassWeight XGBoost 0.433237 0.433973 -0.000736
ClassWeight XGBoost 0.433499 0.434494 -0.000995


In [ ]:
import pandas as pd

# ============================================================
# 가중치 설정
# ============================================================
WEIGHTS = {
    "Test_Recall"  : 0.2,
    "Test_F1"      : 0.6,
    "Test_PR_AUC" : 0.2,
}

# ============================================================
# 데이터 로드
# ============================================================
df = pd.read_csv("Summary_CV_Test_MultiFeature_try.csv")

# ============================================================
# 가중 점수 계산
# ============================================================
df["Weighted_Score"] = sum(
    df[metric] * weight
    for metric, weight in WEIGHTS.items()
).round(4)

# ============================================================
# 출력 컬럼 정의
# ============================================================
ID_COLS     = ["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"]
SCORE_COLS  = ["Weighted_Score", "Test_F1", "Test_Recall", "Test_PR_AUC","Test_ROC_AUC"]
GAP_COLS    = ["Gap_F1", "Gap_Recall", "Gap_PR_AUC","Gap_ROC_AUC"]
DISPLAY_COLS = ID_COLS + SCORE_COLS + GAP_COLS

weight_str = " + ".join(f"{m}×{w}" for m, w in WEIGHTS.items())

# ============================================================
# 1) 전체 TOP 10
# ============================================================
top10 = (
    df[DISPLAY_COLS]
    .sort_values("Weighted_Score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1

print("=" * 100)
print(f"★ 전체 최적 조합 TOP 10  ({weight_str})")
print(f"   가중치 합계: {sum(WEIGHTS.values()):.1f}")
print("=" * 100)
print(top10.to_string())

# ============================================================
# 2) 피처셋별 TOP 3
# ============================================================
print("\n" + "=" * 100)
print("★ 피처셋별 최적 조합 TOP 3")
print("=" * 100)

for feature_set, group in df.groupby("FeatureSet"):
    feature_file = group["FeatureFile"].iloc[0]
    n_feat       = group["N_Features"].iloc[0]

    top3 = (
        group[DISPLAY_COLS]
        .sort_values("Weighted_Score", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )
    top3.index += 1

    print(f"\n  [{feature_set}]  파일: {feature_file}  |  피처 수: {n_feat}개")
    print(top3[["Method", "SMOTE_Ratio", "Model"] + SCORE_COLS + GAP_COLS].to_string())

# ============================================================
# 3) 피처셋별 1등 비교 요약
# ============================================================
print("\n" + "=" * 100)
print("★ 피처셋별 1등 요약 비교")
print("=" * 100)

best_per_set = (
    df.sort_values("Weighted_Score", ascending=False)
    .groupby("FeatureSet", sort=False)
    .first()
    .reset_index()
)[DISPLAY_COLS]

print(best_per_set.to_string(index=False))

★ 전체 최적 조합 TOP 10  (Test_Recall×0.2 + Test_F1×0.6 + Test_PR_AUC×0.2)
   가중치 합계: 1.0
      FeatureSet                   FeatureFile  N_Features       Method SMOTE_Ratio     Model  Weighted_Score   Test_F1  Test_Recall  Test_PR_AUC  Test_ROC_AUC    Gap_F1  Gap_Recall  Gap_PR_AUC  Gap_ROC_AUC
1  top55_dedup45  lasso_features_top55--45.csv          45  ClassWeight           -   XGBoost          0.5231  0.430987     0.870044     0.452537      0.959680 -0.001503   -0.223481   -0.036473    -0.002722
2  top65_dedup52  lasso_features_top65--52.csv          52  ClassWeight           -   XGBoost          0.5211  0.447797     0.817181     0.444975      0.958748 -0.009340   -0.135328   -0.028423     0.000146
3  top60_dedup49  lasso_features_top60--49.csv          49  ClassWeight           -   XGBoost          0.5208  0.447101     0.823789     0.439142      0.959194 -0.006780   -0.136569   -0.018827    -0.000259
4  top50_dedup43  lasso_features_top50--43.csv          43  ClassWeight           -   XG

# 임계값 조정

In [4]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = r'13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
YEAR_COL     = "회계년도"

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012

RECALL_MIN = 0.8   # Recall 하한 조건


# ============================================================
# 2. Best Parameters
# ============================================================

best_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "verbosity": 0,
    "scale_pos_weight": pos_weight
}

# ============================================================
# 3. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()
best_params["scale_pos_weight"] = pos_weight

print("=" * 65)
print(f"피처 수      : {len(use_features)}개")
print(f"pos_weight   : {pos_weight:.4f}")
print("=" * 65)


# ============================================================
# 4. Expanding Window CV fold 생성
# ============================================================

def make_expanding_folds(df, year_col, fold_val_years, train_start):
    folds = []
    for val_year in fold_val_years:
        train_idx = df.index[
            (df[year_col] >= train_start) & (df[year_col] < val_year)
        ]
        val_idx = df.index[df[year_col] == val_year]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx, val_year))
    return folds

folds = make_expanding_folds(train_full, YEAR_COL, FOLD_VAL_YEARS, TRAIN_START)


# ============================================================
# 5. Threshold 최적화 함수
# ============================================================

def find_best_threshold(y_true, y_prob, recall_min=RECALL_MIN):
    """
    Recall >= recall_min 조건 하에서 F1 최대화하는 threshold 탐색
    0.01 단위로 스캔
    """
    thresholds = np.arange(0.01, 1.0, 0.01)
    results = []

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        pre = precision_score(y_true, y_pred, zero_division=0)
        f1  = f1_score(y_true, y_pred, zero_division=0)
        results.append({"threshold": round(thr, 2), "Recall": rec, "Precision": pre, "F1": f1})

    df_thr = pd.DataFrame(results)

    # Recall >= recall_min 조건 필터링 후 F1 최대
    filtered = df_thr[df_thr["Recall"] >= recall_min]

    if filtered.empty:
        print(f"  [경고] Recall >= {recall_min} 조건을 만족하는 threshold 없음 → 전체에서 F1 최대 선택")
        best_row = df_thr.loc[df_thr["F1"].idxmax()]
    else:
        best_row = filtered.loc[filtered["F1"].idxmax()]

    return best_row, df_thr


def print_metrics(label, y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    print(f"\n  [{label}]  threshold={threshold:.2f}")
    print(f"    F1        : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"    Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"    Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"    ROC_AUC   : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"    PR_AUC    : {average_precision_score(y_true, y_prob):.4f}")


# ============================================================
# 6. CV fold별 결과 + threshold 탐색
# ============================================================

print(f"\n{'='*65}")
print("Expanding Window CV — fold별 결과 + threshold 탐색")
print(f"  조건: Recall >= {RECALL_MIN} 하에서 F1 최대화")
print(f"{'='*65}")

cv_rows        = []
thr_candidates = []   # fold별 최적 threshold 수집 → Test threshold 결정에 활용

for train_idx, val_idx, val_year in folds:

    X_fold_train = train_full.loc[train_idx, use_features]
    y_fold_train = train_full.loc[train_idx, TARGET_COL]
    X_fold_val   = train_full.loc[val_idx,   use_features]
    y_fold_val   = train_full.loc[val_idx,   TARGET_COL]

    # NaN imputation
    imputer      = SimpleImputer(strategy="median")
    X_fold_train = pd.DataFrame(imputer.fit_transform(X_fold_train), columns=use_features)
    X_fold_val   = pd.DataFrame(imputer.transform(X_fold_val),       columns=use_features)

    model = XGBClassifier(**best_params)
    model.fit(X_fold_train, y_fold_train)
    y_prob_val = model.predict_proba(X_fold_val)[:, 1]

    # threshold 탐색
    best_row, _ = find_best_threshold(y_fold_val, y_prob_val, RECALL_MIN)
    opt_thr     = best_row["threshold"]
    thr_candidates.append(opt_thr)

    print(f"\n  Fold (Val={val_year})  →  최적 threshold: {opt_thr:.2f}")
    print_metrics(f"Val {val_year}", y_fold_val, y_prob_val, opt_thr)

    cv_rows.append({
        "Val_Year"          : val_year,
        "Opt_Threshold"     : opt_thr,
        "Val_F1"            : best_row["F1"],
        "Val_Recall"        : best_row["Recall"],
        "Val_Precision"     : best_row["Precision"],
        "Val_ROC_AUC"       : roc_auc_score(y_fold_val, y_prob_val),
        "Val_PR_AUC"        : average_precision_score(y_fold_val, y_prob_val),
    })

cv_df = pd.DataFrame(cv_rows).round(4)

print(f"\n{'='*65}")
print("CV fold별 요약")
print("=" * 65)
print(cv_df.to_string(index=False))
print(f"\n  CV 평균 threshold : {np.mean(thr_candidates):.4f}")
print(f"  CV 평균 F1        : {cv_df['Val_F1'].mean():.4f}")
print(f"  CV 평균 Recall    : {cv_df['Val_Recall'].mean():.4f}")


# ============================================================
# 7. Test 평가 — CV 최적 threshold 중앙값 사용
# ============================================================

# CV fold에서 얻은 threshold의 중앙값을 Test threshold로 사용
test_threshold = float(np.median(thr_candidates))

print(f"\n{'='*65}")
print(f"Test 평가")
print(f"  사용 threshold : median({[round(t,2) for t in thr_candidates]}) = {test_threshold:.2f}")
print("=" * 65)

# 전체 train으로 재학습
imputer_final = SimpleImputer(strategy="median")
X_train_all   = pd.DataFrame(imputer_final.fit_transform(train_full[use_features]), columns=use_features)
X_test_imp    = pd.DataFrame(imputer_final.transform(test[use_features]),            columns=use_features)

final_model = XGBClassifier(**best_params)
final_model.fit(X_train_all, y_train_full)
y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]

# Test threshold 추가 탐색 (참고용)
best_test_row, thr_df = find_best_threshold(y_test, y_prob_test, RECALL_MIN)

print_metrics("Test (CV median threshold)", y_test, y_prob_test, test_threshold)
print_metrics("Test (Test 자체 최적 threshold — 참고용)", y_test, y_prob_test, best_test_row["threshold"])


# ============================================================
# 8. Threshold별 전체 스캔 테이블 출력 (상위 구간만)
# ============================================================

print(f"\n{'='*65}")
print(f"Test threshold 스캔 (Recall >= {RECALL_MIN} 구간)")
print("=" * 65)
scan = thr_df[thr_df["Recall"] >= RECALL_MIN].sort_values("F1", ascending=False).head(15)
print(scan.to_string(index=False))


# ============================================================
# 9. 저장
# ============================================================

cv_df.to_csv("Threshold_CV_results.csv", index=False, encoding="utf-8-sig")
thr_df.to_csv("Threshold_scan_test.csv", index=False, encoding="utf-8-sig")

test_summary = {
    "CV_median_threshold"  : test_threshold,
    "Test_F1"              : f1_score(y_test, (y_prob_test >= test_threshold).astype(int), zero_division=0),
    "Test_Recall"          : recall_score(y_test, (y_prob_test >= test_threshold).astype(int), zero_division=0),
    "Test_Precision"       : precision_score(y_test, (y_prob_test >= test_threshold).astype(int), zero_division=0),
    "Test_ROC_AUC"         : roc_auc_score(y_test, y_prob_test),
    "Test_PR_AUC"          : average_precision_score(y_test, y_prob_test),
    "Best_Test_threshold"  : best_test_row["threshold"],
    "Best_Test_F1"         : best_test_row["F1"],
    "Best_Test_Recall"     : best_test_row["Recall"],
}
pd.DataFrame([test_summary]).round(4).to_csv(
    "Threshold_test_summary.csv", index=False, encoding="utf-8-sig"
)

print(f"\n저장 완료")
print(f"  → Threshold_CV_results.csv     (fold별 최적 threshold + 성능)")
print(f"  → Threshold_scan_test.csv      (Test 전체 threshold 스캔)")
print(f"  → Threshold_test_summary.csv   (최종 Test 성능 요약)")
print("=" * 65)

피처 수      : 45개
pos_weight   : 25.6455

Expanding Window CV — fold별 결과 + threshold 탐색
  조건: Recall >= 0.8 하에서 F1 최대화

  Fold (Val=2016)  →  최적 threshold: 0.25

  [Val 2016]  threshold=0.25
    F1        : 0.3582
    Recall    : 0.8108
    Precision : 0.2299
    ROC_AUC   : 0.9536
    PR_AUC    : 0.4580

  Fold (Val=2017)  →  최적 threshold: 0.24

  [Val 2017]  threshold=0.24
    F1        : 0.3986
    Recall    : 0.8000
    Precision : 0.2654
    ROC_AUC   : 0.9568
    PR_AUC    : 0.3959

  Fold (Val=2018)  →  최적 threshold: 0.36

  [Val 2018]  threshold=0.36
    F1        : 0.4337
    Recall    : 0.8000
    Precision : 0.2975
    ROC_AUC   : 0.9587
    PR_AUC    : 0.3685

  Fold (Val=2019)  →  최적 threshold: 0.40

  [Val 2019]  threshold=0.40
    F1        : 0.4085
    Recall    : 0.8208
    Precision : 0.2719
    ROC_AUC   : 0.9530
    PR_AUC    : 0.4657

  Fold (Val=2020)  →  최적 threshold: 0.12

  [Val 2020]  threshold=0.12
    F1        : 0.3121
    Recall    : 0.8100
    Precision : 0